<a href="https://colab.research.google.com/github/kirti73/BIS_LAB/blob/main/1BF24CS140_ACO_LAB3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Load Dataset and Define Fitness Function
We will load the Wine dataset from `sklearn` and define a fitness function. The fitness function evaluates a subset of features by training a classifier (such as a Support Vector Classifier or Random Forest) and calculating its K-Fold cross-validation accuracy.

In [1]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Load dataset
data = load_wine()
X = data.data
y = data.target
feature_names = np.array(data.feature_names)

print(f"Dataset shape: {X.shape}")
print(f"Feature Names: {feature_names}")

def fitness_function(selected_indices, X, y):
    """
    Evaluates the quality of a feature subset.
    Returns the cross-validation score. If no features are selected, returns 0.
    """
    if len(selected_indices) == 0:
        return 0.0

    # Subset the features
    X_subset = X[:, selected_indices]

    # Define classifier
    clf = RandomForestClassifier(n_estimators=50, random_state=42)

    # Calculate cross-validation score
    scores = cross_val_score(clf, X_subset, y, cv=5)
    return scores.mean()

Dataset shape: (178, 13)
Feature Names: ['alcohol' 'malic_acid' 'ash' 'alcalinity_of_ash' 'magnesium'
 'total_phenols' 'flavanoids' 'nonflavanoid_phenols' 'proanthocyanins'
 'color_intensity' 'hue' 'od280/od315_of_diluted_wines' 'proline']


### 2. Ant Colony Optimization (ACO) for Feature Selection
In ACO, ants traverse a graph where nodes represent features. An ant decides to include or exclude a feature based on pheromone levels and heuristic information (like individual feature importance).

We will implement a simplified version of ACO customized for feature selection.

In [2]:
class AntColonyFeatureSelection:
    def __init__(self, X, y, n_ants=10, n_iterations=20, alpha=1.0, beta=1.0, decay=0.1):
        self.X = X
        self.y = y
        self.n_features = X.shape[1]
        self.n_ants = n_ants
        self.n_iterations = n_iterations
        self.alpha = alpha  # Pheromone importance
        self.beta = beta    # Heuristic importance
        self.decay = decay  # Pheromone evaporation rate

        # Initialize pheromone values on features
        self.pheromones = np.ones(self.n_features)

        # Heuristic information: Use individual feature correlation or simple baseline scores
        # For simplicity, we can use a uniform heuristic or individual feature fitness
        self.heuristics = np.zeros(self.n_features)
        for i in range(self.n_features):
            self.heuristics[i] = fitness_function([i], self.X, self.y)

    def run(self):
        best_fitness = -1
        best_features = []

        for iteration in range(self.n_iterations):
            ant_paths = []
            ant_fitnesses = []

            for ant in range(self.n_ants):
                # Calculate probabilities of selecting each feature
                # P = (pheromone^alpha) * (heuristic^beta)
                probabilities = (self.pheromones ** self.alpha) * (self.heuristics ** self.beta)
                # Normalize probabilities to make a selection choice for each feature
                probabilities /= np.sum(probabilities)

                # Each ant decides whether to select each feature independently based on its probability
                selected_features = []
                for i in range(self.n_features):
                    if np.random.rand() < min(0.9, probabilities[i] * self.n_features * 0.5):
                        selected_features.append(i)

                # Evaluate selected subset
                fit = fitness_function(selected_features, self.X, self.y)
                ant_paths.append(selected_features)
                ant_fitnesses.append(fit)

                # Track global best
                if fit > best_fitness:
                    best_fitness = fit
                    best_features = selected_features

            # Evaporate pheromones
            self.pheromones *= (1 - self.decay)

            # Deposit pheromones based on quality of solutions found in this iteration
            for path, fit in zip(ant_paths, ant_fitnesses):
                if len(path) > 0 and fit > 0:
                    # Better fitness deposits more pheromone; divide by size to penalize overly large feature subsets
                    deposit = fit / (len(path) + 1e-5)
                    for feature in path:
                        self.pheromones[feature] += deposit

            print(f"Iteration {iteration + 1}/{self.n_iterations} - Best Fitness: {best_fitness:.4f} (Selected {len(best_features)} features)")

        return best_features, best_fitness

### 3. Run Optimization and Show Selected Features

In [3]:
# Initialize and run the ACO feature selection
acofs = AntColonyFeatureSelection(X, y, n_ants=15, n_iterations=15, alpha=1.0, beta=1.2, decay=0.2)
best_indices, best_score = acofs.run()

# Display final results
print("\n--- Optimization Complete ---")
print(f"Best CV Accuracy: {best_score:.4f}")
print(f"Selected Features: {feature_names[best_indices].tolist()}")
print(f"Reduced feature set size: {len(best_indices)} out of {X.shape[1]}")

Iteration 1/15 - Best Fitness: 0.9663 (Selected 6 features)
Iteration 2/15 - Best Fitness: 0.9722 (Selected 9 features)
Iteration 3/15 - Best Fitness: 0.9722 (Selected 9 features)
Iteration 4/15 - Best Fitness: 0.9722 (Selected 9 features)
Iteration 5/15 - Best Fitness: 0.9722 (Selected 9 features)
Iteration 6/15 - Best Fitness: 0.9776 (Selected 6 features)
Iteration 7/15 - Best Fitness: 0.9776 (Selected 6 features)
Iteration 8/15 - Best Fitness: 0.9776 (Selected 6 features)
Iteration 9/15 - Best Fitness: 0.9776 (Selected 6 features)
Iteration 10/15 - Best Fitness: 0.9778 (Selected 8 features)
Iteration 11/15 - Best Fitness: 0.9778 (Selected 8 features)
Iteration 12/15 - Best Fitness: 0.9778 (Selected 8 features)
Iteration 13/15 - Best Fitness: 0.9778 (Selected 8 features)
Iteration 14/15 - Best Fitness: 0.9778 (Selected 8 features)
Iteration 15/15 - Best Fitness: 0.9778 (Selected 8 features)

--- Optimization Complete ---
Best CV Accuracy: 0.9778
Selected Features: ['alcohol', 'malic_

### 4. Pseudocode for ACO Feature Selection

```text
FUNCTION fitness_function(selected_features):
    IF selected_features is empty:
        RETURN 0
    X_subset = dataset with only selected_features
    score = mean(5-Fold Cross Validation of Classifier(X_subset))
    RETURN score

FUNCTION Run_ACO_Feature_Selection:
    Initialize pheromones for all features to 1.0
    Calculate heuristics for each individual feature using fitness_function([feature_i])
    
    best_fitness = -1
    best_subset = []

    FOR iteration from 1 to max_iterations:
        FOR each ant in population:
            Calculate probability for each feature based on: (pheromone^alpha) * (heuristic^beta)
            
            For each feature:
                Probabilistically decide to select/exclude based on feature's probability
                
            Calculate fitness of current ant's selected subset
            
            IF fitness > best_fitness:
                best_fitness = fitness
                best_subset = current subset
        
        # Update Pheromones
        Evaporate pheromones on all features: pheromones = pheromones * (1 - decay_rate)
        FOR each ant's chosen subset:
            Deposit new pheromone on its selected features proportional to: (fitness / subset_size)
            
    RETURN best_subset, best_fitness
```